# Tech Challenge 3 - Análise e Predição de Atrasos de Voos nos EUA

**Objetivo:** Desenvolver um pipeline completo de ciência de dados para analisar e prever atrasos de voos nos EUA (2015) utilizando técnicas de Machine Learning supervisionado e não supervisionado.

---

## Sumário
1. [Importação de Bibliotecas e Dados](#1)
2. [Exploração dos Dados (EDA)](#2)
3. [Feature Engineering](#3)
4. [Tratamento e Preparação para Modelagem](#4)
5. [Modelagem Supervisionada](#5)
   - 5.1 Classificação: prever se um voo vai atrasar
   - 5.2 Regressão: prever a magnitude do atraso
6. [Modelagem Não Supervisionada](#6)
   - 6.1 Clusterização (K-Means) de companhias aéreas
   - 6.2 Redução de Dimensionalidade (PCA)
7. [Bônus: Detecção de Anomalias](#7)
8. [Conclusões, Limitações e Próximos Passos](#8)


<a id='1'></a>
## 1. Importação de Bibliotecas e Dados

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, RandomForestRegressor,
    GradientBoostingRegressor, IsolationForest,
)
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    mean_absolute_error, mean_squared_error, r2_score,
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

RANDOM_STATE = 42
print('Bibliotecas carregadas com sucesso!')

In [ ]:
# Carregar as bases de dados
# Para esse notebook esperamos: flights.csv, airlines.csv, airports.csv no diretório de execução.
flights = pd.read_csv('flights.csv', low_memory=False)
airlines = pd.read_csv('airlines.csv')
airports = pd.read_csv('airports.csv')

print(f'Flights:  {flights.shape[0]:,} linhas x {flights.shape[1]} colunas')
print(f'Airlines: {airlines.shape[0]} linhas x {airlines.shape[1]} colunas')
print(f'Airports: {airports.shape[0]} linhas x {airports.shape[1]} colunas')

In [ ]:
# Merge limpo: renomeamos a coluna em airlines ANTES do merge para evitar
# conflito de nomes (que na versão anterior gerava AIRLINE_x/AIRLINE_y).
airlines_clean = airlines.rename(columns={
    'IATA_CODE': 'AIRLINE',          # chave de merge
    'AIRLINE':   'AIRLINE_NAME',     # nome legível
})

flights = flights.merge(airlines_clean, on='AIRLINE', how='left')
print('Colunas após merge:', [c for c in flights.columns if 'AIRLINE' in c])
flights.head()

<a id='2'></a>
## 2. Exploração dos Dados (EDA)

### 2.1 Visão Geral

In [ ]:
print('=== INFORMAÇÕES GERAIS DO DATASET DE VOOS ===')
print(f'Período: {flights["MONTH"].min()}/{flights["YEAR"].min()} a {flights["MONTH"].max()}/{flights["YEAR"].max()}')
print(f'Total de voos: {len(flights):,}')
print(f'Companhias aéreas: {flights["AIRLINE"].nunique()}')
print(f'Aeroportos de origem: {flights["ORIGIN_AIRPORT"].nunique()}')
print(f'Aeroportos de destino: {flights["DESTINATION_AIRPORT"].nunique()}')
print(f'\nVoos cancelados: {flights["CANCELLED"].sum():,} ({flights["CANCELLED"].mean()*100:.2f}%)')
print(f'Voos desviados: {flights["DIVERTED"].sum():,} ({flights["DIVERTED"].mean()*100:.2f}%)')

In [ ]:
# Estatísticas descritivas das variáveis numéricas principais
cols_desc = ['DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'DISTANCE', 'SCHEDULED_TIME',
             'ELAPSED_TIME', 'AIR_TIME', 'TAXI_OUT', 'TAXI_IN']
flights[cols_desc].describe().round(2)

### 2.2 Análise de Valores Ausentes

In [ ]:
missing = flights.isnull().sum()
missing_pct = (missing / len(flights) * 100).round(2)
missing_df = pd.DataFrame({'Ausentes': missing, 'Percentual (%)': missing_pct})
missing_df = missing_df[missing_df['Ausentes'] > 0].sort_values('Percentual (%)', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
missing_df['Percentual (%)'].plot(kind='barh', color='coral', ax=ax)
ax.set_xlabel('Percentual de Valores Ausentes (%)')
ax.set_title('Valores Ausentes por Coluna')
plt.tight_layout()
plt.show()

missing_df

**Estratégia de tratamento:**
- **Colunas de causa de atraso** (`AIR_SYSTEM_DELAY`, `WEATHER_DELAY`, etc.) são `NaN` quando aquele tipo de atraso **não ocorreu** — devem ser preenchidas com **0**, não removidas.
- **Voos cancelados** não têm `ARRIVAL_DELAY` registrado e serão removidos para a modelagem.
- **`TAIL_NUMBER`** ausente é ignorado (não usado nos modelos).

In [ ]:
# Preencher NaN das causas de atraso com 0 (semântica: "não houve aquele tipo de atraso").
# Isso corrige o viés nas médias mostradas na seção 2.8.
delay_cause_cols = ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
                    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']
flights[delay_cause_cols] = flights[delay_cause_cols].fillna(0)
print('NaN nas causas de atraso após preenchimento:')
print(flights[delay_cause_cols].isnull().sum())

### 2.3 Distribuição de Atrasos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

dep_delay = flights['DEPARTURE_DELAY'].dropna()
arr_delay = flights['ARRIVAL_DELAY'].dropna()

# Clip para visualização (não modifica dados originais)
dep_clip = dep_delay[(dep_delay >= -30) & (dep_delay <= 120)]
arr_clip = arr_delay[(arr_delay >= -60) & (arr_delay <= 120)]

axes[0].hist(dep_clip, bins=75, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Sem atraso')
axes[0].axvline(x=15, color='orange', linestyle='--', linewidth=1.5, label='Atraso >= 15 min')
axes[0].set_title('Distribuição do Atraso na Partida')
axes[0].set_xlabel('Minutos de Atraso'); axes[0].set_ylabel('Frequência'); axes[0].legend()

axes[1].hist(arr_clip, bins=75, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=1.5, label='Sem atraso')
axes[1].axvline(x=15, color='purple', linestyle='--', linewidth=1.5, label='Atraso >= 15 min')
axes[1].set_title('Distribuição do Atraso na Chegada')
axes[1].set_xlabel('Minutos de Atraso'); axes[1].set_ylabel('Frequência'); axes[1].legend()

plt.tight_layout()
plt.show()

print(f'Atraso na partida > 0 min:  {(dep_delay > 0).sum():,} voos ({(dep_delay > 0).mean()*100:.1f}%)')
print(f'Atraso na partida >= 15 min: {(dep_delay >= 15).sum():,} voos ({(dep_delay >= 15).mean()*100:.1f}%)')
print(f'Atraso na chegada > 0 min:   {(arr_delay > 0).sum():,} voos ({(arr_delay > 0).mean()*100:.1f}%)')
print(f'Atraso na chegada >= 15 min: {(arr_delay >= 15).sum():,} voos ({(arr_delay >= 15).mean()*100:.1f}%)')

### 2.4 Atrasos por Companhia Aérea

In [ ]:
airline_stats = flights.groupby(['AIRLINE', 'AIRLINE_NAME']).agg(
    total_voos=('FLIGHT_NUMBER', 'count'),
    atraso_partida_medio=('DEPARTURE_DELAY', 'mean'),
    atraso_chegada_medio=('ARRIVAL_DELAY', 'mean'),
    pct_cancelado=('CANCELLED', 'mean'),
).reset_index().sort_values('atraso_partida_medio')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
colors = ['green' if x <= 0 else 'orange' if x <= 10 else 'red'
          for x in airline_stats['atraso_partida_medio']]
axes[0].barh(airline_stats['AIRLINE_NAME'], airline_stats['atraso_partida_medio'], color=colors)
axes[0].set_xlabel('Atraso Médio na Partida (min)')
axes[0].set_title('Atraso Médio na Partida por Companhia Aérea')
axes[0].axvline(x=0, color='black', linewidth=0.5)

vol = airline_stats.sort_values('total_voos')
axes[1].barh(vol['AIRLINE_NAME'], vol['total_voos'], color='steelblue')
axes[1].set_xlabel('Total de Voos')
axes[1].set_title('Volume de Voos por Companhia Aérea')

plt.tight_layout(); plt.show()
airline_stats.round(2)

### 2.5 Atrasos por Mês e Dia da Semana

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

monthly = flights.groupby('MONTH')['DEPARTURE_DELAY'].mean()
meses = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
axes[0].bar(meses, monthly.values, color='steelblue', edgecolor='white')
axes[0].set_ylabel('Atraso Médio na Partida (min)')
axes[0].set_title('Atraso Médio por Mês')
axes[0].axhline(y=monthly.mean(), color='red', linestyle='--',
                label=f'Média geral: {monthly.mean():.1f} min')
axes[0].legend()

weekly = flights.groupby('DAY_OF_WEEK')['DEPARTURE_DELAY'].mean()
dias = ['Seg','Ter','Qua','Qui','Sex','Sáb','Dom']
axes[1].bar(dias, weekly.values, color='darkorange', edgecolor='white')
axes[1].set_ylabel('Atraso Médio na Partida (min)')
axes[1].set_title('Atraso Médio por Dia da Semana')
axes[1].axhline(y=weekly.mean(), color='red', linestyle='--',
                label=f'Média geral: {weekly.mean():.1f} min')
axes[1].legend()

plt.tight_layout(); plt.show()

### 2.6 Atrasos por Horário de Partida

In [ ]:
# HOUR = parte inteira do horário (formato HHMM -> HH)
flights['HOUR'] = (flights['SCHEDULED_DEPARTURE'] // 100).astype('Int64')

hourly = flights.groupby('HOUR').agg(
    atraso_medio=('DEPARTURE_DELAY', 'mean'),
    total_voos=('FLIGHT_NUMBER', 'count'),
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.bar(hourly['HOUR'], hourly['total_voos'], color='lightblue', alpha=0.7, label='Total de Voos')
ax1.set_xlabel('Hora Programada de Partida')
ax1.set_ylabel('Total de Voos', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax1.twinx()
ax2.plot(hourly['HOUR'], hourly['atraso_medio'], color='red', linewidth=2, marker='o',
         label='Atraso Médio')
ax2.set_ylabel('Atraso Médio (min)', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('Volume de Voos e Atraso Médio por Hora do Dia')
fig.legend(loc='upper left', bbox_to_anchor=(0.12, 0.88))
plt.tight_layout(); plt.show()

### 2.7 Aeroportos Mais Críticos

In [ ]:
# Top 15 aeroportos de origem com maior atraso médio (mínimo 1.000 voos)
airport_delays = flights.groupby('ORIGIN_AIRPORT').agg(
    atraso_medio=('DEPARTURE_DELAY', 'mean'),
    total_voos=('FLIGHT_NUMBER', 'count'),
).reset_index()
airport_delays = airport_delays[airport_delays['total_voos'] >= 1000]
top_delay = airport_delays.nlargest(15, 'atraso_medio')

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(top_delay['ORIGIN_AIRPORT'], top_delay['atraso_medio'], color='indianred')
ax.set_xlabel('Atraso Médio na Partida (min)')
ax.set_title('Top 15 Aeroportos com Maior Atraso Médio na Partida (mín. 1.000 voos)')
plt.tight_layout(); plt.show()

### 2.8 Causas dos Atrasos



In [ ]:
delay_labels = ['Sistema Aéreo', 'Segurança', 'Companhia Aérea',
                'Aeronave Atrasada', 'Clima']
cause_means = flights[delay_cause_cols].mean()  # agora correto: NaN já é 0

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].bar(delay_labels, cause_means.values,
            color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'])
axes[0].set_ylabel('Contribuição Média (min/voo)')
axes[0].set_title('Contribuição Média de Cada Causa ao Atraso Total')
axes[0].tick_params(axis='x', rotation=30)

proportions = (cause_means / cause_means.sum() * 100).values
axes[1].pie(proportions, labels=delay_labels, autopct='%1.1f%%', startangle=90,
            colors=['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6'])
axes[1].set_title('Proporção das Causas de Atraso (do total minutos)')

plt.tight_layout(); plt.show()

### 2.9 Correlação entre Variáveis Numéricas

In [ ]:
cols_corr = ['DEPARTURE_DELAY', 'ARRIVAL_DELAY', 'DISTANCE', 'SCHEDULED_TIME',
             'TAXI_OUT', 'TAXI_IN', 'AIR_TIME', 'MONTH', 'DAY_OF_WEEK', 'HOUR']

sample_corr = flights[cols_corr].sample(n=min(200_000, len(flights)),
                                        random_state=RANDOM_STATE)
corr_matrix = sample_corr.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Matriz de Correlação - Variáveis Numéricas')
plt.tight_layout(); plt.show()

### 2.10 [BÔNUS] Mapa Geográfico dos Aeroportos

Aproveitando que `airports.csv` traz LAT/LONG, podemos visualizar onde estão os aeroportos com maiores atrasos.

In [ ]:
# Juntar estatísticas de atraso com coordenadas dos aeroportos
airport_geo = airport_delays.merge(
    airports[['IATA_CODE', 'AIRPORT', 'CITY', 'STATE', 'LATITUDE', 'LONGITUDE']],
    left_on='ORIGIN_AIRPORT', right_on='IATA_CODE', how='left'
).dropna(subset=['LATITUDE', 'LONGITUDE'])

# Filtrar para EUA continental para visualização (excluir AK, HI e territórios)
mainland = airport_geo[
    (airport_geo['LATITUDE'].between(24, 50)) &
    (airport_geo['LONGITUDE'].between(-130, -65))
]

fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(
    mainland['LONGITUDE'], mainland['LATITUDE'],
    s=mainland['total_voos'] / 200,
    c=mainland['atraso_medio'],
    cmap='RdYlGn_r', alpha=0.7, edgecolors='black', linewidth=0.5,
)
plt.colorbar(scatter, label='Atraso Médio na Partida (min)')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Aeroportos dos EUA (continental) — tamanho = volume, cor = atraso médio')

# Rotular top 10 aeroportos com mais voos
for _, row in mainland.nlargest(10, 'total_voos').iterrows():
    ax.annotate(row['ORIGIN_AIRPORT'], (row['LONGITUDE'], row['LATITUDE']),
                fontsize=9, fontweight='bold', ha='center', va='bottom',
                xytext=(0, 5), textcoords='offset points')

plt.tight_layout(); plt.show()

<a id='3'></a>
## 3. Feature Engineering

Aqui criamos variáveis derivadas que o EDA sugere serem relevantes:
- **`PERIOD_OF_DAY`** - período (madrugada / manhã / tarde / noite) a partir de `HOUR`.
- **`SEASON`** — estação do ano (no hemisfério norte, baseado em `MONTH`).
- **`IS_WEEKEND`** — flag para fim de semana.
- **`IS_HOLIDAY`** — flag para feriado federal dos EUA em 2015 (e período de Thanksgiving / Natal).

In [ ]:
# 3.1 Período do dia
def get_period(h):
    if pd.isna(h): return np.nan
    if 0 <= h < 6:   return 'madrugada'
    if 6 <= h < 12:  return 'manha'
    if 12 <= h < 18: return 'tarde'
    return 'noite'

flights['PERIOD_OF_DAY'] = flights['HOUR'].apply(get_period)

# 3.2 Estação do ano (hemisfério norte)
season_map = {12: 'inverno', 1: 'inverno', 2: 'inverno',
              3: 'primavera', 4: 'primavera', 5: 'primavera',
              6: 'verao',    7: 'verao',    8: 'verao',
              9: 'outono',  10: 'outono',  11: 'outono'}
flights['SEASON'] = flights['MONTH'].map(season_map)

# 3.3 Fim de semana
flights['IS_WEEKEND'] = (flights['DAY_OF_WEEK'].isin([6, 7])).astype(int)

# 3.4 Feriados federais EUA 2015 + períodos de alto tráfego
holidays_2015 = pd.to_datetime([
    '2015-01-01',  # New Year
    '2015-01-19',  # MLK Day
    '2015-02-16',  # Presidents Day
    '2015-05-25',  # Memorial Day
    '2015-07-03',  # Independence Day (observado)
    '2015-07-04',  # Independence Day
    '2015-09-07',  # Labor Day
    '2015-10-12',  # Columbus Day
    '2015-11-11',  # Veterans Day
    '2015-11-25',  # véspera Thanksgiving (alto tráfego)
    '2015-11-26',  # Thanksgiving
    '2015-11-27',  # Black Friday
    '2015-11-29',  # retorno de Thanksgiving
    '2015-12-23',  # véspera Natal
    '2015-12-24',  # Natal Eve
    '2015-12-25',  # Christmas
    '2015-12-31',  # New Year's Eve
])

flights['FLIGHT_DATE'] = pd.to_datetime(
    flights[['YEAR', 'MONTH', 'DAY']].rename(columns=str.lower),
    errors='coerce'
)
flights['IS_HOLIDAY'] = flights['FLIGHT_DATE'].isin(holidays_2015).astype(int)

# Sanity check
print('Distribuição PERIOD_OF_DAY:')
print(flights['PERIOD_OF_DAY'].value_counts(dropna=False))
print('\nDistribuição SEASON:')
print(flights['SEASON'].value_counts())
print(f'\nFim de semana: {flights["IS_WEEKEND"].mean()*100:.1f}% dos voos')
print(f'Feriados:      {flights["IS_HOLIDAY"].mean()*100:.1f}% dos voos')

In [ ]:
# Validar que as novas features carregam sinal
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

period_order = ['madrugada', 'manha', 'tarde', 'noite']
period_delay = flights.groupby('PERIOD_OF_DAY')['DEPARTURE_DELAY'].mean().reindex(period_order)
axes[0].bar(period_order, period_delay.values, color='teal', edgecolor='white')
axes[0].set_ylabel('Atraso Médio na Partida (min)')
axes[0].set_title('Atraso por Período do Dia')

holiday_delay = flights.groupby('IS_HOLIDAY')['DEPARTURE_DELAY'].mean()
axes[1].bar(['Dia normal', 'Feriado'], holiday_delay.values, color=['steelblue', 'crimson'])
axes[1].set_ylabel('Atraso Médio na Partida (min)')
axes[1].set_title('Atraso em Feriados vs Dias Normais')

plt.tight_layout(); plt.show()

<a id='4'></a>
## 4. Tratamento e Preparação para Modelagem

In [ ]:
print(f'Shape original: {flights.shape}')

# 1. Remover voos cancelados (não têm informações de atraso)
df = flights[flights['CANCELLED'] == 0].copy()
print(f'Após remover cancelados: {df.shape}')

# 2. Remover linhas sem informação de atraso (target principal)
df = df.dropna(subset=['ARRIVAL_DELAY', 'DEPARTURE_DELAY'])
print(f'Após remover sem atraso registrado: {df.shape}')

# 3. Target binário: atraso significativo (>= 15 min na chegada)
df['IS_DELAYED'] = (df['ARRIVAL_DELAY'] >= 15).astype(int)

print(f'\nDistribuição do target IS_DELAYED:')
print((df['IS_DELAYED'].value_counts(normalize=True) * 100).round(2))
print(f'\nShape final para modelagem: {df.shape}')

<a id='5'></a>
## 5. Modelagem Supervisionada

### 5.1 Classificação: Prever se um Voo Vai Atrasar

**Target:** `IS_DELAYED` (1 = atraso >= 15 min na chegada).

**Features:**
- Numéricas: `MONTH`, `DAY_OF_WEEK`, `HOUR`, `DISTANCE`, `SCHEDULED_TIME`, `IS_WEEKEND`, `IS_HOLIDAY`.
- Categóricas (one-hot): `AIRLINE`, `PERIOD_OF_DAY`, `SEASON`.
- Categórica de alta cardinalidade (target encoding): `ORIGIN_AIRPORT`.

**Por que não LabelEncoder em AIRLINE?** Modelos lineares interpretariam o código como ordinal (AA "menor" que UA), o que é falso para variáveis nominais.

**Por que target encoding em ORIGIN_AIRPORT?** Cerca de 300 aeroportos: one-hot daria 300 colunas. Target encoding com smoothing condensa a informação em 1 coluna, mas precisa ser computado **apenas no treino** para não vazar.

**Algoritmos comparados:**
1. Regressão Logística (com `class_weight='balanced'`)
2. Random Forest (com `class_weight='balanced'`)

In [ ]:
# Amostra para viabilizar treino em tempo razoável
sample_size = 300_000
df_sample = df.sample(n=min(sample_size, len(df)), random_state=RANDOM_STATE)

# Selecionar features
numeric_features = ['MONTH', 'DAY_OF_WEEK', 'HOUR', 'DISTANCE',
                    'SCHEDULED_TIME', 'IS_WEEKEND', 'IS_HOLIDAY']
categorical_features = ['AIRLINE', 'PERIOD_OF_DAY', 'SEASON']
high_card_feature = 'ORIGIN_AIRPORT'

# One-hot encoding para baixa cardinalidade (substitui LabelEncoder)
df_encoded = pd.get_dummies(
    df_sample[numeric_features + categorical_features + [high_card_feature, 'IS_DELAYED']],
    columns=categorical_features, drop_first=True
)

# Remover NaN remanescentes
df_encoded = df_encoded.dropna()
print(f'Após dropna: {df_encoded.shape}')

y = df_encoded['IS_DELAYED'].astype(int)
X = df_encoded.drop(columns=['IS_DELAYED'])
print(f'Features totais (antes do target encoding): {X.shape[1]}')

In [ ]:
# Split treino/teste — stratify para preservar proporção de classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Target encoding com smoothing — calculado SÓ no treino para evitar leakage
GLOBAL_MEAN = y_train.mean()
SMOOTHING = 100  # quanto maior, mais conservador (encolhe para a média global)

train_airport_stats = pd.DataFrame({
    'ORIGIN_AIRPORT': X_train['ORIGIN_AIRPORT'],
    'target': y_train.values,
}).groupby('ORIGIN_AIRPORT')['target'].agg(['mean', 'count']).reset_index()

train_airport_stats['encoded'] = (
    (train_airport_stats['mean'] * train_airport_stats['count'] + GLOBAL_MEAN * SMOOTHING)
    / (train_airport_stats['count'] + SMOOTHING)
)
airport_encoding = dict(zip(train_airport_stats['ORIGIN_AIRPORT'],
                            train_airport_stats['encoded']))

X_train['ORIGIN_ENC'] = X_train['ORIGIN_AIRPORT'].map(airport_encoding).fillna(GLOBAL_MEAN)
X_test['ORIGIN_ENC']  = X_test['ORIGIN_AIRPORT'].map(airport_encoding).fillna(GLOBAL_MEAN)

X_train = X_train.drop(columns=['ORIGIN_AIRPORT'])
X_test  = X_test.drop(columns=['ORIGIN_AIRPORT'])

print(f'Features finais: {X_train.shape[1]}')
print(f'Treino: {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,}')
print(f'Taxa de atraso no treino: {y_train.mean()*100:.2f}%')

In [ ]:
# Normalizar (apenas para Logistic Regression; árvores não precisam)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

#### Modelo 1: Regressão Logística (com balanceamento de classes)

In [ ]:
print('=== REGRESSÃO LOGÍSTICA (class_weight=balanced) ===')
lr = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE,
                        class_weight='balanced', n_jobs=-1)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print(f'Acurácia: {accuracy_score(y_test, y_pred_lr):.4f}')
print(f'AUC-ROC:  {roc_auc_score(y_test, y_prob_lr):.4f}')
print(classification_report(y_test, y_pred_lr, target_names=['Sem Atraso', 'Atrasado']))

#### Modelo 2: Random Forest (com balanceamento de classes)

In [ ]:
print('=== RANDOM FOREST CLASSIFIER (class_weight=balanced) ===')
rf = RandomForestClassifier(
    n_estimators=200, max_depth=15, min_samples_leaf=20,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print(f'Acurácia: {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'AUC-ROC:  {roc_auc_score(y_test, y_prob_rf):.4f}')
print(classification_report(y_test, y_pred_rf, target_names=['Sem Atraso', 'Atrasado']))

#### Comparação Visual dos Modelos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# 1. Curva ROC
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
axes[0].plot(fpr_lr, tpr_lr, lw=2,
             label=f'Logistic Regression (AUC={roc_auc_score(y_test, y_prob_lr):.3f})')
axes[0].plot(fpr_rf, tpr_rf, lw=2,
             label=f'Random Forest (AUC={roc_auc_score(y_test, y_prob_rf):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('Curva ROC'); axes[0].legend()

# 2. Matriz de Confusão - LogReg
sns.heatmap(confusion_matrix(y_test, y_pred_lr), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Sem Atraso', 'Atrasado'],
            yticklabels=['Sem Atraso', 'Atrasado'], ax=axes[1])
axes[1].set_title('Matriz de Confusão - Logistic Regression')
axes[1].set_ylabel('Real'); axes[1].set_xlabel('Predito')

# 3. Matriz de Confusão - RF
sns.heatmap(confusion_matrix(y_test, y_pred_rf), annot=True, fmt='d', cmap='Greens',
            xticklabels=['Sem Atraso', 'Atrasado'],
            yticklabels=['Sem Atraso', 'Atrasado'], ax=axes[2])
axes[2].set_title('Matriz de Confusão - Random Forest')
axes[2].set_ylabel('Real'); axes[2].set_xlabel('Predito')

plt.tight_layout(); plt.show()

#### Análise de Threshold (Precision-Recall)

Com `class_weight='balanced'` o threshold default (0.5) prioriza recall. Em produção podemos querer outro ponto de operação por exemplo, alertar passageiros com base no quanto queremos minimizar falsos alarmes.

In [ ]:
# Curva Precision-Recall para o Random Forest
prec, rec, thr = precision_recall_curve(y_test, y_prob_rf)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(thr, prec[:-1], label='Precision', linewidth=2)
ax.plot(thr, rec[:-1], label='Recall', linewidth=2)
ax.axvline(x=0.5, color='gray', linestyle='--', label='Threshold default (0.5)')
ax.set_xlabel('Threshold de Classificação')
ax.set_ylabel('Score')
ax.set_title('Trade-off Precision-Recall — Random Forest')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Ponto de operação onde precision = recall
idx = np.argmin(np.abs(prec[:-1] - rec[:-1]))
print(f'Threshold de equilíbrio: {thr[idx]:.3f}')
print(f'  Precision: {prec[idx]:.3f} | Recall: {rec[idx]:.3f}')

#### Importância das Features (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X_train.columns)
importances = importances.sort_values(ascending=True).tail(20)

fig, ax = plt.subplots(figsize=(10, 7))
importances.plot(kind='barh', color='forestgreen', ax=ax)
ax.set_xlabel('Importância'); ax.set_title('Top 20 Features Mais Importantes')
plt.tight_layout(); plt.show()

#### Validação Cruzada (5-fold)

Para garantir que os resultados não dependem do split específico, implementeio validação cruzada estratificada.

In [ ]:
print('=== VALIDAÇÃO CRUZADA (5-fold estratificada) ===')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Para evitar custo proibitivo, validamos numa amostra menor
cv_idx = X_train.sample(n=min(100_000, len(X_train)), random_state=RANDOM_STATE).index
X_cv = X_train.loc[cv_idx]; y_cv = y_train.loc[cv_idx]
X_cv_scaled = scaler.transform(X_cv)

scores_lr = cross_val_score(LogisticRegression(max_iter=1000, class_weight='balanced',
                                               random_state=RANDOM_STATE),
                            X_cv_scaled, y_cv, cv=cv, scoring='roc_auc', n_jobs=-1)
scores_rf = cross_val_score(RandomForestClassifier(n_estimators=100, max_depth=15,
                                                   class_weight='balanced',
                                                   random_state=RANDOM_STATE, n_jobs=-1),
                            X_cv, y_cv, cv=cv, scoring='roc_auc', n_jobs=-1)

print(f'Logistic Regression — AUC: {scores_lr.mean():.4f} (+/- {scores_lr.std()*2:.4f})')
print(f'Random Forest       — AUC: {scores_rf.mean():.4f} (+/- {scores_rf.std()*2:.4f})')

#### Comparação com Split Temporal

Em produção, treinamos em dados passados e prevemos para o futuro. O split aleatório acima vaza informação temporal. Aqui treinamos nos meses 1-9 e testamos em 10-12.

In [ ]:
# Split temporal
mask_train_t = df_sample['MONTH'] <= 9
mask_test_t  = df_sample['MONTH'] >= 10

# Recriar X com features (mesma engenharia)
df_temporal = pd.get_dummies(
    df_sample[numeric_features + categorical_features + [high_card_feature, 'IS_DELAYED', 'MONTH']],
    columns=categorical_features, drop_first=True
).dropna()

X_train_t = df_temporal[df_temporal['MONTH'] <= 9].drop(columns=['IS_DELAYED', 'MONTH'])
X_test_t  = df_temporal[df_temporal['MONTH'] >= 10].drop(columns=['IS_DELAYED', 'MONTH'])
y_train_t = df_temporal[df_temporal['MONTH'] <= 9]['IS_DELAYED'].astype(int)
y_test_t  = df_temporal[df_temporal['MONTH'] >= 10]['IS_DELAYED'].astype(int)

# Re-fit target encoding usando só o treino temporal
gm_t = y_train_t.mean()
stats_t = pd.DataFrame({'a': X_train_t['ORIGIN_AIRPORT'], 't': y_train_t.values}) \
            .groupby('a')['t'].agg(['mean', 'count']).reset_index()
stats_t['enc'] = (stats_t['mean'] * stats_t['count'] + gm_t * SMOOTHING) \
                 / (stats_t['count'] + SMOOTHING)
enc_t = dict(zip(stats_t['a'], stats_t['enc']))
X_train_t['ORIGIN_ENC'] = X_train_t['ORIGIN_AIRPORT'].map(enc_t).fillna(gm_t)
X_test_t['ORIGIN_ENC']  = X_test_t['ORIGIN_AIRPORT'].map(enc_t).fillna(gm_t)
X_train_t = X_train_t.drop(columns=['ORIGIN_AIRPORT'])
X_test_t  = X_test_t.drop(columns=['ORIGIN_AIRPORT'])

rf_t = RandomForestClassifier(n_estimators=200, max_depth=15, min_samples_leaf=20,
                              class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf_t.fit(X_train_t, y_train_t)
y_prob_rf_t = rf_t.predict_proba(X_test_t)[:, 1]

print(f'AUC (split aleatório):  {roc_auc_score(y_test, y_prob_rf):.4f}')
print(f'AUC (split temporal):   {roc_auc_score(y_test_t, y_prob_rf_t):.4f}')
print()
print('Se a queda for grande, o modelo se aproveita de padrões temporais que')
print('não generalizam para o futuro — sinal de overfitting temporal.')

### 5.2 Regressão: Prever a Magnitude do Atraso

**Decisão importante :** prevendo `ARRIVAL_DELAY` para **todos os voos** (inclusive os pontuais e adiantados), não apenas para `> 0`. Por quê?
- Em produção, ninguém sabe a priori se o voo vai atrasar. Treinar só nos atrasados criaria um modelo inutilizável sem rodar o classificador antes.
- Manter o universo completo torna o modelo honesto sobre sua incerteza.

Aplicamos um clip em [-30, 300] para remover outliers absurdos sem distorcer a maior parte da distribuição.

**Algoritmos comparados:**
1. Random Forest Regressor
2. Gradient Boosting Regressor

In [ ]:
# Preparar dados de regressão (mesmas features da classificação)
df_reg = df.sample(n=min(200_000, len(df)), random_state=RANDOM_STATE).copy()

df_reg_enc = pd.get_dummies(
    df_reg[numeric_features + categorical_features + [high_card_feature, 'ARRIVAL_DELAY']],
    columns=categorical_features, drop_first=True
).dropna()

# Clip de outliers
df_reg_enc = df_reg_enc[df_reg_enc['ARRIVAL_DELAY'].between(-30, 300)]

y_reg = df_reg_enc['ARRIVAL_DELAY']
X_reg = df_reg_enc.drop(columns=['ARRIVAL_DELAY'])

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=RANDOM_STATE
)

# Target encoding (apenas com base no treino)
gm_r = y_train_r.mean()
stats_r = pd.DataFrame({'a': X_train_r['ORIGIN_AIRPORT'], 't': y_train_r.values}) \
            .groupby('a')['t'].agg(['mean', 'count']).reset_index()
stats_r['enc'] = (stats_r['mean'] * stats_r['count'] + gm_r * SMOOTHING) \
                 / (stats_r['count'] + SMOOTHING)
enc_r = dict(zip(stats_r['a'], stats_r['enc']))
X_train_r['ORIGIN_ENC'] = X_train_r['ORIGIN_AIRPORT'].map(enc_r).fillna(gm_r)
X_test_r['ORIGIN_ENC']  = X_test_r['ORIGIN_AIRPORT'].map(enc_r).fillna(gm_r)
X_train_r = X_train_r.drop(columns=['ORIGIN_AIRPORT'])
X_test_r  = X_test_r.drop(columns=['ORIGIN_AIRPORT'])

print(f'Treino: {X_train_r.shape[0]:,} | Teste: {X_test_r.shape[0]:,}')
print(f'Atraso médio (treino):  {y_train_r.mean():.2f} min')
print(f'Atraso mediano (treino): {y_train_r.median():.2f} min')

In [ ]:
# Modelo 1: Random Forest Regressor
print('=== RANDOM FOREST REGRESSOR ===')
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_leaf=20,
                               random_state=RANDOM_STATE, n_jobs=-1)
rf_reg.fit(X_train_r, y_train_r)
y_pred_rf_r = rf_reg.predict(X_test_r)

mae_rf  = mean_absolute_error(y_test_r, y_pred_rf_r)
rmse_rf = np.sqrt(mean_squared_error(y_test_r, y_pred_rf_r))
r2_rf   = r2_score(y_test_r, y_pred_rf_r)
print(f'MAE:  {mae_rf:.2f} min')
print(f'RMSE: {rmse_rf:.2f} min')
print(f'R²:   {r2_rf:.4f}')

In [ ]:
# Modelo 2: Gradient Boosting Regressor
print('=== GRADIENT BOOSTING REGRESSOR ===')
gb_reg = GradientBoostingRegressor(n_estimators=200, max_depth=6, learning_rate=0.1,
                                   random_state=RANDOM_STATE)
gb_reg.fit(X_train_r, y_train_r)
y_pred_gb_r = gb_reg.predict(X_test_r)

mae_gb  = mean_absolute_error(y_test_r, y_pred_gb_r)
rmse_gb = np.sqrt(mean_squared_error(y_test_r, y_pred_gb_r))
r2_gb   = r2_score(y_test_r, y_pred_gb_r)
print(f'MAE:  {mae_gb:.2f} min')
print(f'RMSE: {rmse_gb:.2f} min')
print(f'R²:   {r2_gb:.4f}')

In [ ]:
# Comparação visual
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

n_show = 5000
axes[0].scatter(y_test_r.values[:n_show], y_pred_rf_r[:n_show], alpha=0.2, s=10, color='steelblue')
axes[0].plot([-30, 300], [-30, 300], 'r--', lw=2)
axes[0].set_xlabel('Atraso Real (min)'); axes[0].set_ylabel('Atraso Predito (min)')
axes[0].set_title(f'Random Forest (R²={r2_rf:.3f}, MAE={mae_rf:.1f} min)')

axes[1].scatter(y_test_r.values[:n_show], y_pred_gb_r[:n_show], alpha=0.2, s=10, color='darkorange')
axes[1].plot([-30, 300], [-30, 300], 'r--', lw=2)
axes[1].set_xlabel('Atraso Real (min)'); axes[1].set_ylabel('Atraso Predito (min)')
axes[1].set_title(f'Gradient Boosting (R²={r2_gb:.3f}, MAE={mae_gb:.1f} min)')

plt.tight_layout(); plt.show()

results_reg = pd.DataFrame({
    'Modelo': ['Random Forest', 'Gradient Boosting'],
    'MAE (min)':  [mae_rf, mae_gb],
    'RMSE (min)': [rmse_rf, rmse_gb],
    'R²':         [r2_rf, r2_gb],
}).round(4)
results_reg

<a id='6'></a>
## 6. Modelagem Não Supervisionada

### 6.1 Clusterização (K-Means) - Companhias Aéreas

Agrupei companhias por padrões operacionais: atraso médio, distância média, taxa de cancelamento e volume de voos.

In [ ]:
airline_features = flights.groupby(['AIRLINE', 'AIRLINE_NAME']).agg(
    atraso_partida_medio=('DEPARTURE_DELAY', 'mean'),
    atraso_chegada_medio=('ARRIVAL_DELAY', 'mean'),
    distancia_media=('DISTANCE', 'mean'),
    tempo_voo_medio=('SCHEDULED_TIME', 'mean'),
    taxa_cancelamento=('CANCELLED', 'mean'),
    total_voos=('FLIGHT_NUMBER', 'count'),
    taxi_out_medio=('TAXI_OUT', 'mean'),
    taxi_in_medio=('TAXI_IN', 'mean'),
).reset_index()

feature_cols_cluster = ['atraso_partida_medio', 'atraso_chegada_medio', 'distancia_media',
                        'tempo_voo_medio', 'taxa_cancelamento', 'total_voos',
                        'taxi_out_medio', 'taxi_in_medio']

scaler_cluster = StandardScaler()
X_cluster = scaler_cluster.fit_transform(airline_features[feature_cols_cluster])

airline_features[['AIRLINE_NAME'] + feature_cols_cluster].round(2)

In [ ]:
# Método do Cotovelo
inertias = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(K_range, inertias, 'bo-', lw=2)
ax.set_xlabel('Número de Clusters (K)'); ax.set_ylabel('Inércia')
ax.set_title('Método do Cotovelo — K-Means')
plt.tight_layout(); plt.show()

In [ ]:
# Aplicar K-Means com K=3
km = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
airline_features['Cluster'] = km.fit_predict(X_cluster)

# Visualizar em 2D via PCA
pca_2d = PCA(n_components=2)
X_pca_2d = pca_2d.fit_transform(X_cluster)

fig, ax = plt.subplots(figsize=(10, 7))
scatter = ax.scatter(X_pca_2d[:, 0], X_pca_2d[:, 1],
                     c=airline_features['Cluster'], cmap='Set1',
                     s=200, edgecolors='black', linewidth=1)
for i, name in enumerate(airline_features['AIRLINE_NAME']):
    ax.annotate(name, (X_pca_2d[i, 0], X_pca_2d[i, 1]),
                fontsize=7, ha='center', va='bottom',
                xytext=(0, 8), textcoords='offset points')

ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% variância)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% variância)')
ax.set_title('Clusterização das Companhias Aéreas (K-Means, K=3)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout(); plt.show()

In [ ]:
# Interpretação dos clusters
cluster_summary = airline_features.groupby('Cluster')[feature_cols_cluster].mean().round(2)
print('=== PERFIL MÉDIO DOS CLUSTERS ===')
print(cluster_summary.to_string())

for c in sorted(airline_features['Cluster'].unique()):
    names = airline_features.loc[airline_features['Cluster'] == c, 'AIRLINE_NAME'].tolist()
    print(f'\nCluster {c}: {names}')

### 6.2 Redução de Dimensionalidade (PCA) - Voos


In [ ]:
pca_features = ['MONTH', 'DAY_OF_WEEK', 'HOUR', 'ARRIVAL_DELAY',
                'DISTANCE', 'SCHEDULED_TIME', 'TAXI_OUT', 'TAXI_IN']

df_pca_sample = df[pca_features].dropna().sample(n=min(100_000, len(df)),
                                                 random_state=RANDOM_STATE)

scaler_pca = StandardScaler()
X_pca_full = scaler_pca.fit_transform(df_pca_sample)

pca = PCA()
X_pca_transformed = pca.fit_transform(X_pca_full)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
            pca.explained_variance_ratio_, color='steelblue', alpha=0.8)
axes[0].set_xlabel('Componente Principal'); axes[0].set_ylabel('Variância Explicada')
axes[0].set_title('Variância Explicada por Componente')

axes[1].plot(range(1, len(pca.explained_variance_ratio_) + 1),
             np.cumsum(pca.explained_variance_ratio_), 'bo-', lw=2)
axes[1].axhline(y=0.9, color='red', linestyle='--', label='90% variância')
axes[1].set_xlabel('Número de Componentes'); axes[1].set_ylabel('Variância Cumulativa')
axes[1].set_title('Variância Cumulativa Explicada'); axes[1].legend()

plt.tight_layout(); plt.show()

cumvar = np.cumsum(pca.explained_variance_ratio_)
n_90 = int(np.argmax(cumvar >= 0.9)) + 1
print(f'Componentes necessários para 90% da variância: {n_90}')
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {v*100:.2f}% (acumulado: {cumvar[i]*100:.2f}%)')

In [ ]:
# Loadings dos componentes principais
loadings = pd.DataFrame(
    pca.components_[:4].T,
    columns=['PC1', 'PC2', 'PC3', 'PC4'],
    index=pca_features
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Loadings dos Componentes Principais (PC1 a PC4)')
plt.tight_layout(); plt.show()

In [ ]:
# Visualização 2D do PCA colorida por nível de atraso
fig, ax = plt.subplots(figsize=(10, 7))

delay_category = pd.cut(
    df_pca_sample['ARRIVAL_DELAY'],
    bins=[-np.inf, 0, 15, 60, np.inf],
    labels=['Adiantado', 'Leve (0-15min)', 'Moderado (15-60min)', 'Severo (>60min)']
)

colors_map = {'Adiantado': '#2ecc71', 'Leve (0-15min)': '#f1c40f',
              'Moderado (15-60min)': '#e67e22', 'Severo (>60min)': '#e74c3c'}

for cat in ['Adiantado', 'Leve (0-15min)', 'Moderado (15-60min)', 'Severo (>60min)']:
    mask = delay_category == cat
    ax.scatter(X_pca_transformed[mask, 0], X_pca_transformed[mask, 1],
               alpha=0.15, s=5, label=cat, color=colors_map[cat])

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Projeção PCA dos Voos (colorido por nível de atraso)')
ax.legend(markerscale=5)
plt.tight_layout(); plt.show()

<a id='7'></a>
## 7. Detecção de Anomalias com Isolation Forest

Identificando voos com perfil operacional atípico (combinação incomum de duração, taxi time, atraso etc.). Esses casos podem ser úteis para investigação: voos com problemas operacionais raros, possíveis erros de dados ou eventos fora do padrão.

In [ ]:
# Features para detecção (sem o target ARRIVAL_DELAY explícito —
# queremos encontrar anomalias operacionais, não apenas voos atrasados).
anomaly_features = ['DEPARTURE_DELAY', 'TAXI_OUT', 'TAXI_IN',
                    'AIR_TIME', 'DISTANCE', 'SCHEDULED_TIME']

df_anom = df[anomaly_features].dropna().sample(n=min(100_000, len(df)),
                                               random_state=RANDOM_STATE)

iso = IsolationForest(contamination=0.01, random_state=RANDOM_STATE, n_jobs=-1)
df_anom['anomaly'] = iso.fit_predict(df_anom)
df_anom['anomaly_score'] = iso.score_samples(df_anom[anomaly_features])

n_anom = (df_anom['anomaly'] == -1).sum()
print(f'Anomalias detectadas: {n_anom:,} ({n_anom/len(df_anom)*100:.2f}%)')

# Comparar distribuição de atraso na partida em anomalias vs normais
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
normal = df_anom[df_anom['anomaly'] == 1]
anom   = df_anom[df_anom['anomaly'] == -1]

axes[0].hist(normal['DEPARTURE_DELAY'].clip(-30, 300), bins=50,
             alpha=0.6, label=f'Normais (n={len(normal):,})', color='steelblue')
axes[0].hist(anom['DEPARTURE_DELAY'].clip(-30, 300), bins=50,
             alpha=0.6, label=f'Anomalias (n={len(anom):,})', color='red')
axes[0].set_xlabel('Atraso na Partida (min)'); axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição do Atraso — Anomalias vs Normais')
axes[0].legend()

# Tabela resumo
summary = df_anom.groupby('anomaly')[anomaly_features].mean().round(1)
summary.index = ['Anomalia', 'Normal']
sns.heatmap(summary.T, annot=True, fmt='.1f', cmap='RdYlBu_r', ax=axes[1])
axes[1].set_title('Média das Features — Anomalias vs Normais')

plt.tight_layout(); plt.show()
print('\nResumo (média das features por grupo):')
print(summary.T)

<a id='8'></a>
## 8. Conclusões, Limitações e Próximos Passos

> **⚠️ Placeholders a preencher após executar o notebook:** os números abaixo entre `<...>` devem ser substituídos pelos valores reais que saírem da sua execução. O texto está estruturado para destacar o que reportar.

### Principais Conclusões

**Exploração dos Dados:**
- O dataset contém **~5,8 milhões de voos** domésticos nos EUA em 2015, operados por **14 companhias aéreas** em ~300 aeroportos.
- Cerca de **<X>%** dos voos atrasaram mais de 15 min na chegada (preencher com `df['IS_DELAYED'].mean()*100`).
- **Padrões temporais:** atrasos crescem ao longo do dia (período `noite` >> `madrugada`) e em feriados/verão. Voos de feriado têm atraso médio ~**<X>** min vs **<Y>** min em dias normais.
- **Causas:** após o tratamento correto de NaN, as principais contribuições ao atraso total foram <ordenar do gráfico 2.8>.

**Modelagem Supervisionada — Classificação:**
- **Regressão Logística:** AUC = **<X>**, recall na classe "Atrasado" = **<Y>** (com `class_weight='balanced'`).
- **Random Forest:** AUC = **<X>**, recall na classe "Atrasado" = **<Y>**.
- **Validação cruzada (5-fold):** AUC RF = **<X> ± <Y>**, mostrando estabilidade do modelo.
- **Split temporal:** AUC = **<X>** — comparar com o split aleatório (**<Y>**); diferença grande sinalizaria que o modelo se aproveita de padrões temporais não-generalizáveis.
- **Features mais importantes:** `ORIGIN_ENC` (target encoded), `HOUR`, `DISTANCE`, e indicadores de período/estação.

**Modelagem Supervisionada — Regressão:**
- Prever a **magnitude exata** do atraso é difícil sem dados em tempo real (clima, tráfego). R² obtido foi modesto: **<X>** (RF) vs **<Y>** (GB).
- MAE de **~<X> minutos** indica que, em média, erramos a previsão por essa magnitude — útil como estimativa grosseira, não como SLA.

**Modelagem Não Supervisionada:**
- **K-Means (K=3)** separou as companhias em perfis distintos — preencher com a lista de companhias por cluster da seção 6.1.
- **PCA:** **<N>** componentes explicam 90% da variância. PC1 captura predominantemente <variáveis dominantes — ver loadings>.

**Bônus — Detecção de Anomalias:**
- Isolation Forest identificou **~1%** dos voos como anômalos. A média de atraso e taxi time desses casos é substancialmente maior — candidatos a investigação operacional.

### Limitações

1. **Único ano (2015):** padrões podem variar com eventos climáticos atípicos ou mudanças regulatórias.
2. **Sem dados externos:** clima em tempo real, congestionamento, status da aeronave anterior — todas seriam features de alto valor preditivo.
3. **Amostragem:** usamos 300k voos para classificação. Modelo treinado no dataset completo poderia capturar mais sinal.
4. **Target encoding com smoothing fixo:** o valor de smoothing (100) foi escolhido por convenção; vale tunar.
5. **`SCHEDULED_DEPARTURE` ainda é usado para derivar `HOUR`:** essa é informação disponível no momento da reserva, então não há leak — mas para o regressor de magnitude do atraso, `TAXI_OUT` (que usamos no PCA e em anomalias) só é conhecido após o voo decolar. Vale revisar se algum modelo usa features de tempo-real indevidamente.

### Próximos Passos

1. **Feature engineering avançado:**
   - Histórico do aeroporto (atraso médio nas últimas N horas).
   - Histórico da aeronave (atraso do voo anterior do mesmo `TAIL_NUMBER`).
   - Integrar dados de clima do NOAA.

2. **Modelos mais robustos:**
   - XGBoost / LightGBM com tuning via Optuna.
   - Calibração de probabilidades (Platt scaling / isotonic) para usar o threshold de forma confiável.

3. **Validação temporal rigorosa:** `TimeSeriesSplit` em todas as métricas reportadas.

4. **Deploy:** API FastAPI com `/predict_delay_proba`, monitoramento de drift e log de predições para retreinamento contínuo.

5. **Análise causal:** ir além de correlação — quais intervenções operacionais reduziriam atrasos? (e.g., aumentar buffer de turnaround em determinados aeroportos).

6. **Dashboard:** Streamlit/Power BI para que stakeholders consultem aeroportos/companhias críticos.

---
